# Mathématiques appliquées à l'Intelligence Artificielle
## 📝 Jour 3 : Probabilités et classification
## ✅ TP Corrigé

2e année Bachelor Informatique

---

> © 2025 Mohamed EL AFRIT
> Ce contenu est distribué sous licence [Creative Commons BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.fr)
> [www.mohamedelafrit.com](https://www.mohamedelafrit.com)


## ⚠️ Solutions !


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# DATASET FIL ROUGE — 30 développeurs
# ============================================================
# Colonnes : experience, nb_langages, niveau_etudes,
#            taille_entreprise, remote, salaire
noms_colonnes = ["experience", "nb_langages", "niveau_etudes",
                 "taille_entreprise", "remote", "salaire"]

data = np.array([
    [ 0, 2, 3,  150,  80, 28.0],  # 0  junior
    [ 1, 1, 2,   50, 100, 26.5],  # 1  autodidacte
    [ 1, 3, 4,  800,  40, 35.0],  # 2  sortie ecole
    [ 2, 2, 3,  200,  60, 32.0],  # 3
    [ 2, 4, 4, 3000,  20, 38.0],  # 4
    [ 0, 1, 1,   30,  50, 25.0],  # 5  debutant
    [ 3, 3, 3,  100,  80, 34.5],  # 6
    [ 1, 5, 4,   15, 100, 42.0],  # 7  junior polyglotte
    [ 4, 3, 3,  500,  60, 38.5],  # 8
    [ 5, 4, 4, 1200,  40, 44.0],  # 9
    [ 5, 3, 3,  300,  80, 41.0],  # 10
    [ 6, 5, 4, 2000,  30, 48.0],  # 11
    [ 6, 4, 3,  150,  90, 43.5],  # 12
    [ 7, 5, 4,  800,  50, 50.0],  # 13
    [ 7, 3, 2,   80,  70, 39.0],  # 14 exp mais Bac+2
    [ 4, 6, 5,  400, 100, 47.0],  # 15
    [ 8, 4, 4, 1500,  40, 51.0],  # 16
    [ 5, 2, 3, 4000,  10, 40.0],  # 17
    [ 9, 5, 4,  600,  60, 52.0],  # 18
    [10, 6, 4, 2500,  30, 55.0],  # 19
    [10, 4, 3,  200,  80, 48.5],  # 20
    [11, 5, 5, 1000,  50, 58.0],  # 21
    [12, 7, 4, 3500,  20, 60.0],  # 22
    [ 9, 3, 3,   60,  90, 42.0],  # 23 senior sous-paye
    [13, 6, 4,  800,  70, 62.0],  # 24
    [11, 4, 4, 5000,  10, 54.0],  # 25
    [15, 8, 4, 1200,  50, 65.0],  # 26
    [17, 6, 5, 2000,  40, 68.0],  # 27
    [20, 7, 4,  500,  60, 72.0],  # 28
    [18, 5, 3,  100,  80, 58.0],  # 29
])

experience = data[:, 0]
nb_langages = data[:, 1]
salaire = data[:, 5]
salaire_eleve = (salaire > 45).astype(int)
n = len(data)

print(f"Dataset charge : {n} developpeurs, {data.shape[1]} colonnes")
print(f"Classe 0 (<=45k) : {np.sum(salaire_eleve==0)}, Classe 1 (>45k) : {np.sum(salaire_eleve==1)}")


---
## Exercice 1 — Probabilités


In [ ]:
p1 = np.mean(salaire_eleve); print(f"P(sal>45k) = {p1:.2f}")
mask_exp10 = experience > 10; print(f"P(sal>45k | exp>10) = {np.mean(salaire_eleve[mask_exp10]):.2f}")
mask_lang = nb_langages >= 5; print(f"P(sal>45k | lang≥5) = {np.mean(salaire_eleve[mask_lang]):.2f}")


---
## Exercice 2 — Frontière


In [ ]:
X = data[:, :2]; y = salaire_eleve
w = np.array([0.5, 0.8]); b = -4.0
scores = X @ w + b; y_pred = (scores > 0).astype(int)
acc = np.mean(y_pred == y); print(f"Accuracy = {acc:.0%}")

fig, ax = plt.subplots(figsize=(8, 6))
for c, color, label in [(0,'#e74c3c','≤ 45k'),(1,'#2ecc71','> 45k')]:
    mask = y == c; ax.scatter(X[mask,0], X[mask,1], c=color, s=70, label=label, edgecolors='white')
x1 = np.linspace(-1, 21, 100); ax.plot(x1, -(w[0]/w[1])*x1 - b/w[1], 'k--', lw=2)
ax.set_xlim(-1,21); ax.set_ylim(0,9); ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


---
## Exercice 3 — Perceptron


In [ ]:
X = data[:, :2]; y = salaire_eleve
w = np.zeros(2); b = 0.0; alpha = 0.1
for epoch in range(100):
    errs = 0
    for i in range(n):
        z = w @ X[i] + b; yp = 1 if z > 0 else 0
        if yp != y[i]:
            err = y[i] - yp; w += alpha*err*X[i]; b += alpha*err; errs += 1
    if epoch < 5 or epoch % 20 == 19:
        acc = np.mean([(1 if (w@X[j]+b)>0 else 0)==y[j] for j in range(n)])
        print(f"Epoch {epoch+1:3d} : erreurs={errs}, acc={acc:.0%}")


---
## Exercice 4 — vs sklearn


In [ ]:
from sklearn.linear_model import Perceptron as SkP
X = data[:, :2]; y = salaire_eleve
# From scratch
w_fs = np.zeros(2); b_fs = 0.0
for _ in range(100):
    for i in range(n):
        z=w_fs@X[i]+b_fs; yp=1 if z>0 else 0
        if yp!=y[i]: e=y[i]-yp; w_fs+=0.1*e*X[i]; b_fs+=0.1*e
acc_fs = np.mean([(1 if (w_fs@X[i]+b_fs)>0 else 0)==y[i] for i in range(n)])

clf = SkP(max_iter=100, eta0=0.1, random_state=42); clf.fit(X, y); acc_sk = clf.score(X, y)
print(f"From scratch : w=[{w_fs[0]:.3f},{w_fs[1]:.3f}], b={b_fs:.3f}, acc={acc_fs:.0%}")
print(f"sklearn      : w=[{clf.coef_[0][0]:.3f},{clf.coef_[0][1]:.3f}], b={clf.intercept_[0]:.3f}, acc={acc_sk:.0%}")


---
## Exercice 5 — XOR + bonus


In [ ]:
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]]); y_xor = np.array([0,1,1,0])
w = np.zeros(2); b = 0.0
for ep in range(200):
    for i in range(4):
        z=w@X_xor[i]+b; yp=1 if z>0 else 0
        if yp!=y_xor[i]: e=y_xor[i]-yp; w+=0.1*e*X_xor[i]; b+=0.1*e
acc = np.mean([(1 if (w@X_xor[i]+b)>0 else 0)==y_xor[i] for i in range(4)])
print(f"XOR standard : accuracy = {acc:.0%} (ne converge pas)")

# Bonus : ajout de x3 = x1*x2
X_aug = np.column_stack([X_xor, X_xor[:,0]*X_xor[:,1]])
w3 = np.zeros(3); b3 = 0.0
for ep in range(100):
    errs = 0
    for i in range(4):
        z=w3@X_aug[i]+b3; yp=1 if z>0 else 0
        if yp!=y_xor[i]: e=y_xor[i]-yp; w3+=0.1*e*X_aug[i]; b3+=0.1*e; errs+=1
    if errs == 0: print(f"Converge à l'époque {ep+1} !"); break
acc2 = np.mean([(1 if (w3@X_aug[i]+b3)>0 else 0)==y_xor[i] for i in range(4)])
print(f"XOR augmenté (x3=x1*x2) : accuracy = {acc2:.0%}")
